# Titanic：生存者と非生存者の比較（v020-exp-01）

## 分析計画
対象はKaggle Titanic競技のラベル付きtrain相当の公開データ。Kaggle SDKで検索し、ファイル一覧と構造・人数・ID範囲を確認する。testやgender_submissionの予測ラベルは混ぜない。外部CSVミラーへはフォールバックしない。

1. 出所（owner/dataset slug、ファイル、取得日時、SHA-256）と列定義・前提を記録。欠損、重複、値域・意味的異常を検査する。
2. 生存/非生存の人数、性別、客室等級、年齢、運賃、同乗家族数、乗船港、客室情報の有無を比較。カテゴリには分母付き生存率とWilson 95%区間、連続値には中央値/IQRと分布を用いる。
3. 性別×等級の層別比較、調整ロジスティック回帰、欠損/外れ値/年齢区分の感度分析を行う。p値の羅列ではなく差の大きさと不確実性を重視する。
4. 初回結果を読み直し、重要な未解決点があれば自然言語の追加依頼を原文保存し最大3回実行。選定理由・結果・証拠・残る限界を記録する。
5. カーネル再起動後、全コードセルを上からJupyter MCPで再実行しvisual_audit=Trueとライフサイクル状態を保存する。

## 解釈と実行経路
これは観測された学習用部分標本の記述・関連分析であり因果推論ではない。乗客・家族・同一券で観測が依存する可能性、学習用標本の選択、運賃の測定単位、年齢欠損に注意する。分析とKaggle API呼出しはJupyter MCPのみ。mcp_gateway.run_and_recordはカーネル内から同じカーネルを再入実行するクライアントを提供しないため使用せず、実際のJupyter MCP execute_cellを使う（擬似クライアントやexecによる代用なし）。Notebookの構成・証拠・メタデータの明示的変更にはproject_manager.enqueue_writeを使い、セル実行結果はJupyter MCPが保存する。トークン・認証設定は表示しない。


In [1]:
from pathlib import Path
import os, io, json, hashlib, zipfile, re, warnings
from datetime import datetime, timezone
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from dataclasses import asdict
import numpy as np
import pandas as pd
import nbformat
from IPython.display import display
from ai_data_scientist import project_manager as pm, lifecycle, language_router
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
handle = pm.resolve_project('kaggle-v020-kaggle-exp-01-titanic')
pm.ensure_notebook(handle)
data_dir = pm.ensure_data_dir(handle)
RUN_ID = 'v020-exp-01'
lifecycle.register_run(RUN_ID, handle.notebook_path)
phase_log = []
@contextmanager
def phase(label, writing=False):
    if lifecycle.is_cancel_requested(RUN_ID):
        lifecycle.mark_failed(RUN_ID)
        raise RuntimeError('キャンセル要求により停止')
    start = lifecycle.mark_write_start if writing else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if writing else lifecycle.mark_execution_end
    start(RUN_ID)
    phase_log.append({'phase': label, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        raise
    finally:
        end(RUN_ID)
        phase_log.append({'phase': label, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
def safe_api(fn):
    failure = None
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        try:
            result = fn()
        except Exception as exc:
            failure = type(exc).__name__
    if failure:
        raise RuntimeError('Kaggle API失敗（詳細は認証情報保護のため非表示）: ' + failure) from None
    return result
print('run_id:', RUN_ID, 'language:', language_router.detect_language('生存者と非生存者の違いを日本語で調べる'))
print('Notebook:', handle.notebook_path.relative_to(ROOT))
print(asdict(lifecycle.get_run_status(RUN_ID)))


run_id: v020-exp-01 language: ja
Notebook: benchmark/projects/kaggle-v020-kaggle-exp-01-titanic/notebooks/kaggle-v020-kaggle-exp-01-titanic.ipynb
{'run_id': 'v020-exp-01', 'state': 'running', 'active_cell_executions': 0, 'pending_notebook_writes': 0, 'locks_held': 0, 'notebook_path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-01-titanic/notebooks/kaggle-v020-kaggle-exp-01-titanic.ipynb', 'reason': None}


In [4]:
lifecycle.mark_completed(RUN_ID)
RUN_ID = 'v020-exp-01-followup'
lifecycle.register_run(RUN_ID, handle.notebook_path)
from ai_data_scientist import data_definition, analysis_assumptions, data_quality, anomaly_detection, sensitivity, eda, visualization, insight_engine, notebook_audit
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image
print('followup run_id:', RUN_ID)
print('追加依頼は既存記録0回から開始。最大3回。')


followup run_id: v020-exp-01-followup
追加依頼は既存記録0回から開始。最大3回。


In [5]:
with phase('Kaggle公開データ検索・ファイル一覧確認'):
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    safe_api(lambda: api.authenticate())
    candidates = safe_api(lambda: api.dataset_list(search='titanic', page=1))
    refs = [str(d.ref) for d in candidates]
    print('Kaggle検索結果:', refs)
    chosen_slug = None
    chosen_files = None
    preferred = ['heptapod/titanic', 'yasserh/titanic-dataset']
    ordered = [s for s in preferred if s in refs] + [s for s in refs if s not in preferred]
    for slug in ordered[:12]:
        listing = safe_api(lambda slug=slug: api.dataset_list_files(slug))
        file_names = [f.name for f in listing.files]
        print('ファイル一覧:', slug, file_names)
        train_files = [f for f in file_names if f.rsplit('/', 1)[-1].lower() == 'train.csv']
        if train_files:
            chosen_slug, chosen_files = slug, train_files
            break
    if chosen_slug is None:
        raise RuntimeError('対応する公開train.csvをKaggle APIで確認できず停止。外部ソースへのフォールバックなし。')
    file_name = chosen_files[0]
    raw_dir = data_dir / 'kaggle_raw'
    raw_dir.mkdir(parents=True, exist_ok=True)
    safe_api(lambda: api.dataset_download_file(chosen_slug, file_name, path=str(raw_dir), force=True, quiet=True))
    plain = raw_dir / Path(file_name).name
    zipped = raw_dir / (Path(file_name).name + '.zip')
    if plain.is_file() and zipfile.is_zipfile(plain):
        archive = plain
    elif zipped.is_file():
        archive = zipped
    else:
        archive = None
    if archive is not None:
        with zipfile.ZipFile(archive) as z:
            matches = [n for n in z.namelist() if Path(n).name == Path(file_name).name]
            if len(matches) != 1:
                raise RuntimeError('ZIP内の対応CSVを一意に選べず停止')
            payload = z.read(matches[0])
    elif plain.is_file():
        payload = plain.read_bytes()
    else:
        raise RuntimeError('Kaggle SDKの取得ファイルが確認できず停止')
    csv_path = data_dir / 'train.csv'
    csv_path.write_bytes(payload)
    from ai_data_scientist.ingestion import ingest, SourceSpec
    ingestion_result = ingest(SourceSpec(kind='csv', location=str(csv_path)))
    df = ingestion_result.dataframe
    assert not ingestion_result.truncated
    print('取得データ型:', type(df).__name__)
    provenance = {'competition_reference': 'https://www.kaggle.com/competitions/titanic', 'owner_dataset_slug': chosen_slug, 'retrieved_file': file_name, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(payload).hexdigest(), 'bytes': len(payload)}
    print(json.dumps(provenance, ensure_ascii=False, indent=2))
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    display(df.head(3).drop(columns=['Name', 'Ticket', 'Cabin'], errors='ignore'))


Kaggle検索結果: ['heptapod/titanic', 'brendan45774/test-file', 'yasserh/titanic-dataset', 'azeembootwala/titanic', 'rahulsah06/titanic', 'sakshisatre/titanic-dataset', 'shubhamgupta012/titanic-dataset', 'ibrahimelsayed182/titanic-dataset', 'waqi786/titanic-dataset', 'lovishbansal123/titanic-dataset', 'fossouodonald/titaniccsv', 'amineipad/titanic-dataset', 'zain280/titanic-data-set', 'prkukunoor/TitanicDataset', 'hesh97/titanicdataset-traincsv', 'pavlofesenko/titanic-extended', 'jamesleslie/titanic-cleaned-data', 'ashishkumarjayswal/titanic-datasets', 'smirkxd/titanic-dataset', 'hardikgarg03/titanic']
ファイル一覧: heptapod/titanic ['train_and_test2.csv']
ファイル一覧: yasserh/titanic-dataset ['Titanic-Dataset.csv']
ファイル一覧: brendan45774/test-file ['tested.csv']
ファイル一覧: azeembootwala/titanic ['test_data.csv', 'train_data.csv']
ファイル一覧: rahulsah06/titanic ['gender_submission.csv', 'test.csv', 'train.csv']
取得データ型: DataFrame
{
  "competition_reference": "https://www.kaggle.com/competitions/titanic",
  "own

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.2500,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.9250,S


## 再評価記録（v020-exp-01-followup）
確認時の既存Notebookは3セル（計画、初期設定、Kaggle取得）のみ。初回の比較結果、AI自身の反復依頼、図、証拠manifest、完了監査は存在しなかった。既存セルのsourceとIDを保持し、取得済みの891行12列、owner/slugとSHA-256を起点に不足を補う。未実施を『既存分析で確認済み』とは扱わない。

## AIが生成した追加依頼1（原文・実行前記録）
「生存者と非生存者の違いにまだ答えられていないため、取得済みTitanic train.csvについて列の定義と確度、欠損・重複・意味的異常を記録してください。人数、性別、等級、年齢、運賃、同乗家族数、乗船港、客室情報有無を分母と不確実性付きで比較し、性別×等級の層別、調整ロジスティック回帰、欠損処理・運賃外れ値・年齢区分・同一券依存の感度を確認してください。可視化を保存し、結論を実行済み出力に結び付け、因果や全乗客への一般化は避けてください。」

選定理由：比較結果と関連する検証自体が未実施で、問いへの回答を妨げる。追加依頼1は分析上の反復回数であり、再実行・出力修正・監査は新しい依頼に数えない。追加依頼2・3は結果の再評価後、結論に影響する未解決点があるときだけ実施する。
